In [ ]:
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array


# =========================================================
# LOAD TRAJECTORIES
# =========================================================

trajfiles_bdn = [
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1_f.xtc",
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep2_f.xtc",
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep3_f.xtc",
]

trajfiles_bdq = [
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1_f.xtc",
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep2_f.xtc",
    "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep3_f.xtc",
]

topfile_bdn = "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1.tpr"
topfile_bdq = "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1.tpr"


bdn1 = mda.Universe(topfile_bdn, trajfiles_bdn[0])
bdn2 = mda.Universe(topfile_bdn, trajfiles_bdn[1])
bdn3 = mda.Universe(topfile_bdn, trajfiles_bdn[2])

bdq1 = mda.Universe(topfile_bdq, trajfiles_bdq[0])
bdq2 = mda.Universe(topfile_bdq, trajfiles_bdq[1])
bdq3 = mda.Universe(topfile_bdq, trajfiles_bdq[2])


# =========================================================
# CONTACT FUNCTION (POPE)
# =========================================================

def contacts_timeseries_pope(u, ligand_resid, cutoff1=5.0, cutoff2=10.0):

    ligand = u.select_atoms(f"resid {ligand_resid}")

    # POPE membrane selection
    pope = u.select_atoms("resname POPE and not name H*")

    if len(ligand) == 0:
        raise ValueError(f"No ligand atoms found for resid {ligand_resid}")

    if len(pope) == 0:
        raise ValueError("No POPE atoms found in system")

    contacts_5A = []
    contacts_10A = []

    for ts in u.trajectory:

        d = distance_array(
            ligand.positions,
            pope.positions,
            box=u.dimensions
        )

        contacts_5A.append(np.sum(d < cutoff1))
        contacts_10A.append(np.sum(d < cutoff2))

    return np.array(contacts_5A), np.array(contacts_10A)


# =========================================================
# PIPELINE
# =========================================================

def run_pope_contact_analysis(systems, pairs, outdir):

    os.makedirs(outdir, exist_ok=True)

    for system_name, replicas in systems.items():

        system_dir = os.path.join(outdir, system_name)
        os.makedirs(system_dir, exist_ok=True)

        for lig_name, (lig_resid) in pairs.items():

            print(f"{system_name} | {lig_name}")

            lig_dir = os.path.join(system_dir, lig_name)
            os.makedirs(lig_dir, exist_ok=True)

            all_replicas = []

            for rep_idx, u in enumerate(replicas, start=1):

                contacts_5A, contacts_10A = contacts_timeseries_pope(
                    u,
                    lig_resid
                )

                df = pd.DataFrame({
                    "System": system_name,
                    "Ligand": lig_name,
                    "LigandResid": lig_resid,
                    "Replica": rep_idx,
                    "Frame": np.arange(len(contacts_5A)),
                    "Contacts_5A": contacts_5A,
                    "Contacts_10A": contacts_10A
                })

                all_replicas.append(df)

            combined = pd.concat(all_replicas, ignore_index=True)

            outpath = os.path.join(
                lig_dir,
                "pope_contacts_timeseries.csv"
            )

            combined.to_csv(outpath, index=False)

            print(f"Saved -> {outpath}")


# =========================================================
# INPUT
# =========================================================

systems = {
    "BDN": [bdn1, bdn2, bdn3],
    "BDQ": [bdq1, bdq2, bdq3]
}

pairs = {
    "lig1064": 1064,
    "lig1065": 1065,
    "lig1066": 1066,
    "lig1067": 1067,
    "lig1068": 1068,
    "lig1069": 1069,
    "lig1070": 1070,
}


# =========================================================
# RUN
# =========================================================

run_pope_contact_analysis(
    systems=systems,
    pairs=pairs,
    outdir="/scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/pope_contacts"
)